### pyAPES MLM benchmarking at FI-Var (Värriö subarctic pine forest)

Samuli Launiainen 25.05.26 / testing simulations for year 2015 against above- and sub-canopy measurements




In [ ]:
# setting path
import sys
#sys.path.append('c:\\Repositories\\pyAPES_main')
import os
import importlib
from dotenv import load_dotenv

load_dotenv()
pyAPES_main_folder = os.getenv('pyAPES_main_folder')

sys.path.append(pyAPES_main_folder)
os.chdir(pyAPES_main_folder)  # set working directory so relative file paths resolve correctly
#print(sys.path)

# force iPython re-import modules at each call
%load_ext autoreload
%autoreload 2


### Import modules

In [ ]:
# force always reload parameters
import pyAPES.parameters.mlm_parameters_FI_Var as _varrio_params
importlib.reload(_varrio_params)

from pyAPES.pyAPES_MLM import driver

# import parameter dictionaries
from pyAPES.parameters.mlm_parameters_FI_Var import gpara, cpara, spara # model configuration, canopy parameters, soil parameters
from pyAPES.utils.iotools import read_forcing, read_results,  read_data

# python packages
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

eps = 1e-16


In [ ]:
print(gpara['start_time'], gpara['end_time'])

### Read forcing data and compile input for driver


In [ ]:
forcing = read_forcing(
    forcing_file=gpara['forc_filename'],
    start_time=gpara['start_time'],
    end_time=gpara['end_time'],
    dt=gpara['dt']
)

params = {
    'general': gpara,   # model configuration
    'canopy': cpara,    # planttype, micromet, canopy, bottomlayer parameters
    'soil': spara,      # soil heat and water flow parameters
    'forcing': forcing  # forging data
}

### Run the model

In [ ]:
resultfile, Model = driver(parameters=params,
                           create_ncf=True,
                           result_file= 'Varrio_test2.nc'
                          )

### Read model results

In [ ]:
# read simulation results to xarray dataset
results = read_results(os.path.join(pyAPES_main_folder, resultfile))

# read observations from DK-Sor
datafile = r'forcing\FI-Var\FI-Var_2013_2022.dat'
data = read_data(os.path.join(pyAPES_main_folder, datafile), start_time=gpara['start_time'], end_time=gpara['end_time'])

#results = read_results(os.path.join(pyAPES_main_folder, r'results\Soroe_test.nc'))
#data = read_data(os.path.join(pyAPES_main_folder, datafile), start_time='2019-03-01', end_time='2019-10-30')

In [ ]:
# results = read_results(os.path.join(pyAPES_main_folder, r'results\Soroe_test.nc'))
# datafile = r'forcing\Soroe\DK-Sor_EC_2015-2020.dat'
# data = read_data(os.path.join(pyAPES_main_folder, datafile), start_time='2019-03-01', end_time='2019-10-30')

In [ ]:
print(results)

# print list of all variables with their dimensions:
vars = list(results.data_vars)
for v in vars:
    print(f"{v}: {results[v].dims}")

### Plot some model results and compare with observations


In [ ]:
sim = 0  # in this demo, we have only one simulation (i.e. only one parameter set was used)

# grid variables for plotting
t = results.date  # time
zc = results.canopy_z  # height above ground [m]
zs = results.soil_z  # depth of soil; shown negative [m]


In [ ]:
#plt.plot(t, results.canopy_LAI)

### Soil temperature and moisture
- computed using *pyAPES.soil* submodels 'Water' and 'Heat'
- compare with measurements at similar depths (to be done - now compare with Ts and SWC at 5cm depth)

T_depths (m): [0.02, 0.07, 0.17, 0.35]
swc_depths (m): [0.03, 0.09, 0.13, 0.26]


In [ ]:
%matplotlib qt
var = ['soil_temperature', 'soil_volumetric_liquid_water_content']

lyrs = [2, 5, 10, 18] # layers
#depths = np.array2string(np.asarray(zs[lyrs]), precision=1, separator=', ')
depths = ['{:.2f} m'.format(k) for k in zs[lyrs]]

print(depths)
fig, ax = plt.subplots(len(var), 1, figsize=(10,7))

k = 0
ax[0].plot(t, data['Tsoil0'], 'ko', markersize=3, alpha=0.1, label='Org')
ax[0].plot(t, data['Tsoil1'], 'ko', markersize=3, alpha=0.2, label='1cm')
ax[0].plot(t, data['Tsoil5'], 'ko', markersize=3, alpha=0.4, label='5cm')
ax[0].plot(t, data['Tsoil25'], 'ko', markersize=3, alpha=0.6, label='25cm')

ax[1].plot(t, data['wsoil'], 'ko', markersize=3, alpha=0.1, label='5cm')
ax[1].plot(t, data['wsoil5'], 'ks', markersize=3, alpha=0.2, label='5+cm')
ax[1].plot(t, data['wsoil25'], 'kd', markersize=3, alpha=0.4, label='25cm')
for v in var:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)
    k += 1


### Ecosystem-scale fluxes

- ecosystem - atm. fluxes represent the integrated sinks / sources in soil (soil-module), forestfloor (bottomlayer-module) and vegetation (planttype&canopy -modules)
- comparable to ecosystem - atmosphere exchange
- affected by current model forcing and sub-model instance state (e.g. Planttype and Canopy LAI, phenology, soil temperature, moisture etc.)


In [ ]:

%matplotlib qt
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

def scatter_stats(ax, obs, mod, label='', color='steelblue', marker='o'):
    """Scatter points + regression line with stats. Returns (o, m) of valid pairs."""
    mask = np.isfinite(obs) & np.isfinite(mod)
    o, m = obs[mask], mod[mask]
    if len(o) < 2:
        return o, m
    slope, intercept, r, _, _ = stats.linregress(o, m)
    rmse = np.sqrt(np.mean((m - o) ** 2))
    bias = np.mean(m - o)
    lim = [min(o.min(), m.min()), max(o.max(), m.max())]
    ax.scatter(o, m, s=4, alpha=0.25, color=color, marker=marker, rasterized=True)
    ax.plot(lim, slope * np.array(lim) + intercept, '-', color=color, lw=1.2,
            label=f'{label}  slope={slope:.2f}, R²={r**2:.2f}\nRMSE={rmse:.1f}, bias={bias:.1f}')
    return o, m

def finalise_scatter(ax, all_o, all_m, margin=0.05):
    """Set equal (square) axis limits from combined obs+mod range; draw 1:1 line."""
    combined = np.concatenate([all_o, all_m])
    combined = combined[np.isfinite(combined)]
    if len(combined) == 0:
        return
    lo, hi = combined.min(), combined.max()
    pad = (hi - lo) * margin or abs(lo) * margin or 0.1
    lo -= pad; hi += pad
    ax.plot([lo, hi], [lo, hi], 'k--', lw=0.8, zorder=0)
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)
    ax.set_aspect('equal', adjustable='box')
    ax.set_xlabel('Measured'); ax.set_ylabel('Modelled')
    ax.legend(fontsize=6)

def make_flux_fig(rows, title, figsize):
    """Build a (n_rows × 2) figure; time-series column shares x-axis, scatter column does not."""
    n = len(rows)
    fig, axes = plt.subplots(n, 2, figsize=figsize,
                             gridspec_kw={'width_ratios': [3, 1]})
    if n == 1:
        axes = axes[np.newaxis, :]
    for i in range(1, n):
        axes[i, 0].sharex(axes[0, 0])
    fig.suptitle(title, fontsize=12)
    return fig, axes

# ── canopy grid index nearest to z = 3 m ─────────────────────────────────────
iz3 = int(np.argmin(np.abs(zc.values - 3.0)))
z3  = float(zc[iz3])
print(f'Sub-canopy index: iz3={iz3}, z={z3:.2f} m')

obs_colors = ['k', 'r']

# ── Figure 1: CO2 fluxes ──────────────────────────────────────────────────────
co2_rows = [
    ('NEE (µmol m⁻² s⁻¹)', [
        ('canopy_NEE',      None, data['NEE'],      'NEE above',         '#2e86ab', 'o'),
        ('canopy_co2_flux',  iz3, data['F_c_sub'],  f'Fc_sub {z3:.1f} m','#e9c46a', 's'),
    ]),
    ('GPP (µmol m⁻² s⁻¹)', [
        ('canopy_GPP',      None, data['GPP'],      'GPP above',         '#a23b72', 'o'),
    ]),
    ('Reco (µmol m⁻² s⁻¹)', [
        ('canopy_Reco',     None, data['Reco'],     'Reco above',        '#f18f01', 'o'),
    ]),
]

fig1, axes1 = make_flux_fig(co2_rows, 'CO₂ fluxes', figsize=(12, 3 * len(co2_rows)))

for n, (ylabel, pairs) in enumerate(co2_rows):
    ax_ts, ax_sc = axes1[n, 0], axes1[n, 1]
    all_o, all_m = [], []
    for k, (mod_var, iz, obs_s, lbl, col, mkr) in enumerate(pairs):
        mod_vals = results[mod_var][:, sim, iz].values if iz is not None else results[mod_var][:, sim].values
        obs_arr  = np.array(obs_s, dtype=float)
        ax_ts.plot(t.values, obs_arr, f'{obs_colors[k % 2]}.-', markersize=2, alpha=0.3, label=f'{lbl} meas')
        ax_ts.plot(t.values, mod_vals, color=col, lw=0.8, label=f'{lbl} model')
        o, m = scatter_stats(ax_sc, obs_arr, mod_vals, label=lbl, color=col, marker=mkr)
        all_o.append(o); all_m.append(m)
    finalise_scatter(ax_sc, np.concatenate(all_o), np.concatenate(all_m))
    ax_ts.set_ylabel(ylabel, fontsize=8)
    ax_ts.tick_params(axis='x', labelrotation=20)
    ax_ts.legend(fontsize=7)

axes1[-1, 0].set_xlabel('Date')
fig1.tight_layout()

# ── Figure 2: Energy fluxes ───────────────────────────────────────────────────
energy_rows = [
    ('Rnet (W m⁻²)', [
        ('canopy_Rnet',               None, data['Rn'],     'Rn above',          '#c73e1d', 'o'),
    ]),
    ('H (W m⁻²)', [
        ('canopy_SH',                 None, data['H'],      'H above',           '#3b1f2b', 'o'),
        ('canopy_sensible_heat_flux',  iz3, data['H_sub'],  f'H_sub {z3:.1f} m', '#6a4c93', 's'),
    ]),
    ('LE (W m⁻²)', [
        ('canopy_LE',                 None, data['LE'],     'LE above',          '#44bba4', 'o'),
        ('canopy_latent_heat_flux',    iz3, data['LE_sub'], f'LE_sub {z3:.1f} m','#1a936f', 's'),
    ]),
    ('G (W m⁻²)', [
        ('ffloor_ground_heat',        None, data['G'],      'G',                 '#e94f37', 'o'),
    ]),
]

fig2, axes2 = make_flux_fig(energy_rows, 'Energy fluxes', figsize=(14, 3 * len(energy_rows)))

for n, (ylabel, pairs) in enumerate(energy_rows):
    ax_ts, ax_sc = axes2[n, 0], axes2[n, 1]
    all_o, all_m = [], []
    for k, (mod_var, iz, obs_s, lbl, col, mkr) in enumerate(pairs):
        mod_vals = results[mod_var][:, sim, iz].values if iz is not None else results[mod_var][:, sim].values
        obs_arr  = np.array(obs_s, dtype=float)
        ax_ts.plot(t.values, obs_arr, f'{obs_colors[k % 2]}.-', markersize=2, alpha=0.3, label=f'{lbl} meas')
        ax_ts.plot(t.values, mod_vals, color=col, lw=0.8, label=f'{lbl} model')
        o, m = scatter_stats(ax_sc, obs_arr, mod_vals, label=lbl, color=col, marker=mkr)
        all_o.append(o); all_m.append(m)
    finalise_scatter(ax_sc, np.concatenate(all_o), np.concatenate(all_m))
    ax_ts.set_ylabel(ylabel, fontsize=8)
    ax_ts.tick_params(axis='x', labelrotation=20)
    ax_ts.legend(fontsize=7)

axes2[-1, 0].set_xlabel('Date')
fig2.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(4, 1, figsize=(8,10), sharex=True)
# CO2 fluxes
ax[0].plot(t, data['GPP'], 'k.-', alpha=0.3); ax[0].set_ylabel('GPP (umolm-2s-1)')
ax[0].plot(t, results['canopy_GPP'][:,sim], label='GPP')

ax[0].plot(t, results['ffloor_photosynthesis'], '--')
ax[1].plot(t, data['Reco'], 'k.-', alpha=0.3); ax[1].set_ylabel('RECO (umolm-2s-1)')
ax[1].plot(t, results['canopy_Reco'][:,sim], label='ER')
ax[1].plot(t, results['ffloor_respiration'][:,sim], '--', label='Rffloor')
ax[1].plot(t, results['ffloor_soil_respiration'][:,sim], 'r--', label='Rsoil')
ax[2].plot(t, data['Tsoil5'], 'k.-', alpha=0.3); ax[2].set_ylabel('T (degC)')
ax[3].plot(t, data['wsoil'], 'k.-', alpha=0.3); ax[3].set_ylabel('SWC (m3m-3)')

ax[2].plot(t, results['soil_temperature'][:,sim,lyrs], label=depths)
ax[2].legend(fontsize=8)

var3 = ['soil_volumetric_liquid_water_content']
k = 3
for v in var3:
    ax[k].plot(t, results[v][:,sim,lyrs], label=depths)
    ax[k].set_ylabel(results[v].attrs['units'])
    ax[k].tick_params(axis='x', labelrotation = 20)
    ax[k].legend(fontsize=8)



In [ ]:
print(data.columns)

In [ ]:
# print sub-canopy and forest-floor model output variables
ffloor_vars = [v for v in results.data_vars if 'ffloor' in v]
print("ffloor outputs:", ffloor_vars)


### Ecosystem radiation balance

- net radiation (Rnet) is the sum of net shortwave (SWnet = incoming - reflected) and net longwave (LWnet = incoming - emitted) radiation at canopy top
- computed via models in pyAPES.microclimate.radiation, called iteratively from pyAPES.canopy.mlm_canopy to account for canopy structure and leaf & forest floor temperature
- ecosystem albedo can be computed from radiation profiles at uppermost grid point

In [ ]:
# net radiation components at canopy top
var = ['canopy_Rnet','canopy_SWnet', 'canopy_LWnet']
profs = ['canopy_par_down', 'canopy_par_up', 'canopy_nir_down','canopy_nir_up','canopy_lw_down','canopy_lw_up']

fig, ax = plt.subplots(3, 1, figsize=(8,12), sharex=True)

for v in var:
    ax[0].plot(t, results[v][:, sim], label=v)
ax[0].set_ylabel('W m-2')
ax[0].tick_params(axis='x', labelrotation = 20)
ax[0].legend(fontsize=8)    

# lets plot also partitioning of canopy_SWnet and canopy_LWnet.
# -1 is the index of uppermost gridpoint

for v in ['canopy_par_down', 'canopy_nir_down','canopy_lw_down']: # downward
    ax[1].plot(t, results[v][:,sim,-1], '-', label=v)
for v in ['canopy_par_up', 'canopy_nir_up','canopy_lw_up']: # upward
    ax[1].plot(t, results[v][:,sim,-1], '--', label=v)
ax[1].set_ylabel('W m-2')
ax[1].tick_params(axis='x', labelrotation = 20)
ax[1].legend(fontsize=8)

# canopy albedo
eps = 1e-16

# fraction of par on total SW
f_par = results['canopy_par_down'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + results['canopy_nir_down'][:,sim,-1] + eps)

alb_par = results['canopy_par_up'][:,sim,-1] / (results['canopy_par_down'][:,sim,-1] + eps)
alb_nir = results['canopy_nir_up'][:,sim,-1] / (results['canopy_nir_down'][:,sim,-1] + eps)
alb_sw = f_par * alb_par + (1 - f_par) * alb_nir
alb_sw = np.maximum(0, np.minimum(1.0, alb_sw))

ax[2].plot(t, alb_sw, '-', label='SW albedo')
ax[2].plot(t, alb_par, '-', label='Par albedo')
ax[2].plot(t, alb_nir, '-', label='Nir albedo')

ax[2].tick_params(axis='x', labelrotation = 20)
ax[2].legend(fontsize=8)

In [ ]:
# Explore temperature-related variables in results
temp_vars = [v for v in results.data_vars if 'temp' in v.lower() or 'leaf' in v.lower()]
print("Temperature-related variables:")
for v in temp_vars:
    print(f"  {v}: dims={results[v].dims}, shape={results[v].shape}")

In [ ]:
%matplotlib qt
import numpy as np

# canopy height h = top of planttype[0] LAD profile (highest z where lad > 0)
pt0_lad = list(cpara['planttypes'].values())[0]['lad']
h = float(zc.values[pt0_lad > 0].max())

# find canopy indices closest to z/h = 0.2, 0.6, 1.0
target_ratios = [0.2, 0.6, 1.0]
labels = ['z/h = 0.2 (bottom)', 'z/h = 0.6 (middle)', 'z/h = 1.0 (top)']
colors = ['tab:blue', 'tab:orange', 'tab:green']

idx = [int(np.argmin(np.abs(zc.values - r * h))) for r in target_ratios]
actual_z = [float(zc[i]) for i in idx]
print(f"Canopy height h = {h:.1f} m  (planttype[0]: '{list(cpara['planttypes'].keys())[0]}')")
for r, i, z in zip(target_ratios, idx, actual_z):
    print(f"  z/h = {r} -> index {i}, z = {z:.2f} m")

# compute Tleaf - Tair at each height
Tair = results['canopy_temperature'][:, sim, :]
dT = results['canopy_Tleaf'][:, sim, :] - Tair

fig, axes = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
fig.suptitle('Leaf - air temperature difference (Tleaf - Tair)', fontsize=13)

for ax, i, label, color, z in zip(axes, idx, labels, colors, actual_z):
    ax.plot(t, dT[:, i], color=color, linewidth=0.7, label=f'dT  {label}  (z = {z:.1f} m)')
    ax.axhline(0, color='k', linewidth=0.8, linestyle='--')
    ax.set_ylabel('Tleaf - Tair (deg C)', color=color)
    ax.tick_params(axis='y', labelcolor=color)
    ax.legend(fontsize=9, loc='upper left')
    ax.tick_params(axis='x', labelrotation=20)

    ax2 = ax.twinx()
    ax2.plot(t, Tair[:, i], color='gray', linewidth=0.7, alpha=0.7, label=f'Tair  {label}')
    ax2.set_ylabel('Tair (deg C)', color='gray')
    ax2.tick_params(axis='y', labelcolor='gray')
    ax2.legend(fontsize=9, loc='upper right')

axes[-1].set_xlabel('Date')
fig.tight_layout()
plt.show()


In [ ]:

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from pyAPES.soil.heat import sinusoidal_soil_temperature

# --- grid ---
doys = np.arange(1, 366)
dz = np.array([0.05] * 100)
z = dz / 2 - np.cumsum(dz)          # node elevations, negative downward [m]

# --- model parameters ---
T_mean = 8.0
T_amplitude = 10.0
thermal_diffusivity = 5e-7

# --- compute T(z, doy) ---
T_grid = np.array([
    sinusoidal_soil_temperature(z, doy=d, T_mean=T_mean, T_amplitude=T_amplitude, thermal_diffusivity=thermal_diffusivity,)
    for d in doys
]).T  # shape (n_depths, n_doys)

# --- plot ---
fig, ax = plt.subplots(figsize=(10, 5))

levels = np.arange(-6, 18, 1)
cf = ax.contourf(doys, z, T_grid, levels=levels, cmap='RdBu_r', extend='both')
cs = ax.contour(doys, z, T_grid, levels=levels, colors='k', linewidths=0.5, alpha=0.5)
ax.clabel(cs, levels=levels[::2], fmt='%d°C', fontsize=7, inline=True)

cbar = fig.colorbar(cf, ax=ax, label='Soil temperature (°C)')

ax.set_xlabel('Day of year')
ax.set_ylabel('Depth (m)')
ax.set_title(f'Sinusoidal soil heat-wave model  |  $\\bar{{T}}$={T_mean}°C, $A_0$={T_amplitude}°C')
ax.xaxis.set_major_locator(ticker.MultipleLocator(30))
ax.xaxis.set_minor_locator(ticker.MultipleLocator(10))
ax.set_xlim(1, 365)
ax.set_ylim(z[-1], 0)

plt.tight_layout()
plt.show()


In [ ]:

# Sunlit fraction of foliage: time vs. height
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

sunlit = results['canopy_sunlit_fraction'][:, sim, :].values.T  # shape: (canopy, date)
z = zc.values
T = t.values

fig, ax = plt.subplots(figsize=(12, 5))
pcm = ax.pcolormesh(T, z, sunlit, cmap='YlOrRd', vmin=0, vmax=1, shading='auto')
cbar = fig.colorbar(pcm, ax=ax, label='Sunlit fraction (-)')
ax.set_ylabel('Height (m)')
ax.set_xlabel('Date')
ax.set_title('Sunlit fraction of foliage')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()


In [ ]:

# vol. water content
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np

tmp = results['soil_volumetric_water_content'][:, sim, :].values.T  # shape: (soil, date)
z = zs.values
T = t.values

fig, ax = plt.subplots(figsize=(12, 5))
pcm = ax.pcolormesh(T, z, tmp, cmap='YlOrRd', shading='auto')
cbar = fig.colorbar(pcm, ax=ax, label='m3m-3')
ax.set_ylabel('depth (m)')
ax.set_xlabel('Date')
ax.set_title('vol water content (m3m-3)')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=2))
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

In [ ]:
results.close()